In [ ]:
# ============================================================
# 04 — BASELINE RANKERS (EB-NeRD)
# Single-signal reranking AUC: popularity / recency / content. Popularity dominates; recency collapses.
# Fully self-contained EB-NeRD notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers polars -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
# ---- hardcoded EB-NeRD demo path (fast offline iteration) ----
DEMO = "/kaggle/input/datasets/donbosoc/ebnerd-small"
if not os.path.exists(f"{DEMO}/articles.parquet"):
    DEMO = glob.glob("/kaggle/input/**/ebnerd_demo", recursive=True)[0]
print("DEMO:", DEMO)
PREFIX = "eb"
def pfx(x): return f"{PREFIX}:{x}"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])
def parse_split(base, split):
    """Parse EB-NeRD into unified schema: articles / impressions / history."""
    a = pl.read_parquet(f"{base}/articles.parquet")
    articles = a.select(
        article_id=_prefix(pl.col("article_id")),
        title=pl.col("title").fill_null(""),
        abstract=pl.col("subtitle").fill_null(""),
        body=pl.col("body").fill_null("") if "body" in a.columns else pl.lit(""),
        category=pl.col("category_str").fill_null(""),
        published_time=pl.col("published_time"),
    )
    b = pl.read_parquet(f"{base}/{split}/behaviors.parquet")
    cols = b.columns
    impressions = b.select(
        impression_id=pl.col("impression_id"),
        user_id=_prefix(pl.col("user_id")),
        timestamp=pl.col("impression_time"),
        candidate_ids=pl.col("article_ids_inview").list.eval(_prefix(pl.element())),
        labels=(pl.col("article_ids_clicked").list.eval(_prefix(pl.element()))
                if "article_ids_clicked" in cols else pl.lit(None)),
        session_id=(pl.col("session_id") if "session_id" in cols else pl.lit(0)),
    )
    h = pl.read_parquet(f"{base}/{split}/history.parquet")
    hist = {u: (arts or []) for u, arts in zip(
        h.select(_prefix(pl.col("user_id")))["user_id"].to_list(),
        h["article_id_fixed"].list.eval(_prefix(pl.element())).to_list())}
    return articles, impressions, hist
def build_article_luts(articles_df, raw_parquet_path):
    """Build published_time / pageview / category / text lookups from the raw articles."""
    at = pl.read_parquet(raw_parquet_path)
    pub={pfx(r):p for r,p in zip(at["article_id"].to_list(), at["published_time"].to_list())}
    cat={pfx(r):(c or "") for r,c in zip(at["article_id"].to_list(), at["category_str"].to_list())}
    txt={pfx(r):f"{t or ''} {s or ''}".strip() for r,t,s in zip(
         at["article_id"].to_list(), at["title"].to_list(), at["subtitle"].to_list())}
    def col_or_zero(name):
        if name in at.columns:
            return {pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(), at[name].to_list())}
        return defaultdict(float)
    pv=col_or_zero("total_pageviews"); iv=col_or_zero("total_inviews"); rt=col_or_zero("total_read_time")
    return pub,cat,txt,pv,iv,rt


In [ ]:
art, imp_tr, hist_tr = parse_split(DEMO, "train")
_,   imp_va, hist_va = parse_split(DEMO, "validation")
# article luts from the demo articles.parquet
pub,cat,txt,pv,iv,rt = build_article_luts(art, f"{DEMO}/articles.parquet")
print("articles:", len(pub), "| train imps:", imp_tr.height, "| val imps:", imp_va.height)
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
_aids = list(txt.keys()); _atxt = [txt[i] if txt[i] else "nyhed" for i in _aids]
emb = minilm.encode(_atxt, batch_size=512, normalize_embeddings=True,
                    convert_to_numpy=True, show_progress_bar=True)
emb_by_id = {_aids[i]: emb[i] for i in range(len(_aids))}
id_to_row = {_aids[i]: i for i in range(len(_aids))}
emb_mat = emb
print("multilingual MiniLM encoded:", emb.shape)
def recency(aid,T,tau=24.0):
    p=pub.get(aid)
    if p is None or T is None: return 0.0
    dh=(T-p).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
def user_prof(hist,mh=30):
    ai=hist[-mh:] if hist else []
    cats=[cat.get(x) for x in ai];tot=len([c for c in cats if c])
    cc=Counter(c for c in cats if c);cd={k:v/tot for k,v in cc.items()} if tot else {}
    hv=[emb_by_id[x] for x in ai if x in emb_by_id]
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    return cd,um,hv
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def mrr_i(s,lb):
    o=np.argsort(-s)
    for rank,idx in enumerate(o,1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def ndcg_i(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    ideal=np.sort(lb)[::-1][:k]
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(ideal))
    return float(dcg/idcg) if idcg>0 else 0.0
def iter_impressions(imps_df):
    for row in imps_df.iter_rows(named=True):
        cand=row["candidate_ids"]; labs=row["labels"]; T=row["timestamp"]
        if not cand or not labs: continue
        y=np.array([1 if c in set(labs) else 0 for c in cand])
        if y.sum()==0: continue
        yield row["user_id"], T, cand, y


In [ ]:
# rolling click index (popularity as reranking signal, point-in-time)
click_ev=defaultdict(list); imp_ev=defaultdict(list)
for imps in (imp_tr,):
    for row in imps.iter_rows(named=True):
        T=row["timestamp"]
        for c in (row["candidate_ids"] or []): imp_ev[c].append(T)
        for c in (row["labels"] or []): click_ev[c].append(T)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T); return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))

sig={"popularity":[],"recency":[],"content_e5":[]}
for uid,T,cand,labs in iter_impressions(imp_va):
    cd,um,hv=user_prof(hist_va.get(uid,[]))
    pop=np.array([cnt(click_ev,c,T,24) for c in cand],float)
    rec=np.array([recency(c,T) for c in cand],float)
    e5 =np.array([float(um@emb_by_id[c]) if (um is not None and c in emb_by_id) else 0.0 for c in cand],float)
    for nm,s in [("popularity",pop),("recency",rec),("content_e5",e5)]:
        a=auc_i(s,labs)
        if a is not None: sig[nm].append(a)
print("=== EB-NeRD single-signal reranking AUC (val) ===")
for nm,arr in sig.items(): print(f"  {nm:12s} {np.mean(arr):.4f}")
print("\nPopularity is the strong single signal; recency ~random on the pre-filtered slate.")
